In [12]:
# Exercise 01
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf

# Best Practice: Define a configuration first
conf = SparkConf().setAppName("WordCount").setMaster("spark://namenode:7077")

# FIX: Use getOrCreate() to prevent the "Multiple SparkContexts" error
sc = SparkContext.getOrCreate(conf=conf)

input_file = "/data/BattleCreekDec19_2019.txt"

# Process everything
word_counts = sc.textFile(input_file) \
                .flatMap(lambda line: line.split(" ")) \
                .filter(lambda word: word != "") \
                .map(lambda word: (word.lower().strip(), 1)) \
                .reduceByKey(lambda a, b: a + b) \
                .sortBy(lambda x: x, ascending=False) # Simplified sorting syntax

# Note: .collect() pulls everything into driver memory. 
# If the file is massive, this can cause an OutOfMemory error.
all_results = word_counts.collect()

print(f"Grand Total: {len(all_results)} unique words found.\n")

# Printing thousands of lines might lag your notebook; consider all_results[:100]
for word, count in all_results:
    print(f"'{word}': {count}")

Py4JJavaError: An error occurred while calling o511.defaultParallelism.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
sun.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
sun.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.lang.reflect.Constructor.newInstance(Constructor.java:423)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
py4j.Cl

In [6]:
# Exercise 02
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf

# Best Practice: Define a configuration first
conf = SparkConf().setAppName("AverageWord").setMaster("spark://namenode:7077")

# FIX: Use getOrCreate() to prevent the "Multiple SparkContexts" error
sc = SparkContext.getOrCreate(conf=conf)

input_file = "/data/BattleCreekDec19_2019.txt"
lines = sc.textFile(input_file)

# 2. Extract words and filter out empty strings
words = lines.flatMap(lambda line: line.split(" ")) \
             .filter(lambda word: word != "") \
             .map(lambda word: word.strip().strip('.,!?:;()')) \
             .filter(lambda word: word != "")

# 3. Map each word to (length, 1) to prepare for sum and count
# (word_length, count)
word_stats = words.map(lambda word: (len(word), 1))

# 4. Reduce to get total length and total word count
# aggregate = (total_length, total_word_count)
total_length, total_count = word_stats.reduce(lambda x, y: (x[0] + y[0], x[1] + y[1]))

# 5. Calculate average
if total_count > 0:
    average_length = total_length / total_count
    print(f"Total Words: {total_count}")
    print(f"Total Characters: {total_length}")
    print(f"Average Word Length: {average_length:.2f}")
else:
    print("No words found.")

Total Words: 17830
Total Characters: 74715
Average Word Length: 4.19


In [7]:
# Exercise 03
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf

# Best Practice: Define a configuration first
conf = SparkConf().setAppName("WordCount10").setMaster("spark://namenode:7077")

# FIX: Use getOrCreate() to prevent the "Multiple SparkContexts" error
sc = SparkContext.getOrCreate(conf=conf)
input_file = "/data/BattleCreekDec19_2019.txt"

counts = sc.textFile(input_file) \
           .flatMap(lambda line: line.split(" ")) \
           .filter(lambda word: word != "") \
           .map(lambda word: (word.lower().strip(), 1)) \
           .reduceByKey(lambda a, b: a + b)

for word, count in counts.takeOrdered(10, key=lambda x: -x[1]):
    print(f"'{word}': {count}")
sc.stop()

'the': 698
'and': 494
'i': 491
'to': 422
'you': 392
'a': 361
'they': 316
'of': 308
'we': 251
'in': 196


In [8]:
# Exercise 04
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf

# Best Practice: Define a configuration first
conf = SparkConf().setAppName("JoinOperation").setMaster("spark://namenode:7077")

# FIX: Use getOrCreate() to prevent the "Multiple SparkContexts" error
sc = SparkContext.getOrCreate(conf=conf)

# Define paths (using /data/ as per your notebook convention)
path1 = "/data/BattleCreekDec19_2019.txt"
path2 = "/data/BemidjiSep18_2020.txt"

# Helper function for word count RDD
def get_word_counts(file_path):
    return sc.textFile(file_path) \
             .flatMap(lambda line: line.split(" ")) \
             .map(lambda word: word.lower().strip().strip('.,!?:;()')) \
             .filter(lambda word: word != "") \
             .map(lambda word: (word, 1)) \
             .reduceByKey(lambda a, b: a + b)

# Create two RDDs of (word, count)
rdd1 = get_word_counts(path1)
rdd2 = get_word_counts(path2)

# Perform Inner Join
# Result format: (word, (count_from_file1, count_from_file2))
joined_rdd = rdd1.join(rdd2)

# Sort by combined frequency for display
sorted_joined = joined_rdd.sortBy(lambda x: -(x[1][0] + x[1][1]))

# Collect ALL results to the driver
all_joined_results = sorted_joined.collect()

print(f"Inner Join Results: {len(all_joined_results)} words present in BOTH files")
print("-" * 50)
for word, counts in all_joined_results:
    count1, count2 = counts
    print(f"'{word}': BattleCreek={count1}, Bemidji={count2}")

Inner Join Results: 1072 words present in BOTH files
--------------------------------------------------
'the': BattleCreek=698, Bemidji=612
'and': BattleCreek=494, Bemidji=488
'i': BattleCreek=491, Bemidji=459
'to': BattleCreek=427, Bemidji=402
'you': BattleCreek=457, Bemidji=346
'a': BattleCreek=362, Bemidji=430
'they': BattleCreek=316, Bemidji=292
'of': BattleCreek=311, Bemidji=272
'it': BattleCreek=267, Bemidji=296
'that': BattleCreek=253, Bemidji=250
'we': BattleCreek=251, Bemidji=204
'in': BattleCreek=209, Bemidji=177
'have': BattleCreek=189, Bemidji=170
'but': BattleCreek=165, Bemidji=193
'he': BattleCreek=111, Bemidji=166
'it's': BattleCreek=141, Bemidji=134
'so': BattleCreek=148, Bemidji=126
'said': BattleCreek=142, Bemidji=131
'was': BattleCreek=120, Bemidji=144
'is': BattleCreek=118, Bemidji=133
'do': BattleCreek=126, Bemidji=108
'know': BattleCreek=136, Bemidji=95
'what': BattleCreek=150, Bemidji=78
'don't': BattleCreek=120, Bemidji=100
'people': BattleCreek=110, Bemidji=100

In [9]:
# Exercise 05
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf

# Best Practice: Define a configuration first
conf = SparkConf().setAppName("RemoveDuplicate").setMaster("spark://namenode:7077")

# FIX: Use getOrCreate() to prevent the "Multiple SparkContexts" error
sc = SparkContext.getOrCreate(conf=conf)
input_file = "/data/BattleCreekDec19_2019.txt"

# 1. Load lines and split into words
raw_words = sc.textFile(input_file) \
              .flatMap(lambda line: line.split(" ")) \
              .map(lambda word: word.lower().strip().strip('.,!?:;()')) \
              .filter(lambda word: word != "")

# 2. Get unique words using .distinct()
unique_words_rdd = raw_words.distinct()

# 3. Count the results
total_unique = unique_words_rdd.count()

print(f"Total words (including duplicates): {raw_words.count()}")
print(f"Total unique words (duplicates removed): {total_unique}")

# Show a sample of 10 unique words
print("\nSample of unique words:")
print(unique_words_rdd.take(10))

Total words (including duplicates): 17830
Total unique words (duplicates removed): 2356

Sample of unique words:
['vice', 'good', 'job', 'merry', 'christmas', 'michigan', 'we', 'in', 'was', 'of']
